# Which model predicts rent the best?

`train.py` ships XGBoost. This notebook is the receipt for that choice: I put XGBoost up against a handful of alternatives, evaluated every one the exact same way, on the exact same data, and see whether it actually earns its spot.

The plan:

1. load the same cleaned data `train.py` uses, via `features.py`
2. reuse the same train/test split so nothing leaks
3. candidates: a dumb baseline, two linear models, random forest, gradient boosting, and XGBoost <3
4. score everything in real euros - RMSE / MAE / MAPE / R2
5. confirm the ranking with 5-fold cross-validation (one split is noisy)
6. tune the top contenders and see if the gap holds

In [1]:
import sys

sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot

from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

from features import (clean_data, build_ppm_grid, assign_ppm, trim_ppm_outliers,
                      trim_knn_outliers, impute_medians, FEATURES)

df = pd.read_csv("../data/raw/listings.csv")
df = clean_data(df)
print(len(df), "clean listings")
df[["price", "area", "rooms", "dist_to_center"]].describe()

3557 clean listings


,price,area,rooms,dist_to_center
count,3557.000000,3557.000000,3557.000000,3337.000000
mean,829.215271,68.379252,1.865336,2.628799
std,426.715872,28.345239,0.770391,1.667794
min,100.000000,12.000000,1.000000,0.112507
25%,500.000000,50.000000,1.000000,1.219774
50%,750.000000,65.000000,2.000000,2.527146
75%,1089.000000,80.000000,2.000000,3.889642
max,2000.000000,400.000000,5.000000,11.393182


In [2]:
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = df_train.copy(), df_val.copy()

# same leak-free preprocessing train.py does: the 1% price/m2 trim and the
# median imputation are FIT on train only, then applied to the validation rows.
df_train, ppm_bounds = trim_ppm_outliers(df_train)
df_val, _ = trim_ppm_outliers(df_val, ppm_bounds)
df_train, medians = impute_medians(df_train)
df_val, _ = impute_medians(df_val, medians)

grid_ppm, sector_ppm, global_ppm = build_ppm_grid(df_train)
df_train['knn_ppm'] = assign_ppm(df_train, grid_ppm, sector_ppm, global_ppm)
df_val['knn_ppm'] = assign_ppm(df_val, grid_ppm, sector_ppm, global_ppm)

# and the same market-rate trim train.py does: drop listings asking way off
# their own neighborhood, then recompute the rate from the cleaned market
df_train = trim_knn_outliers(df_train).copy()
df_val = trim_knn_outliers(df_val).copy()
grid_ppm, sector_ppm, global_ppm = build_ppm_grid(df_train)
df_train['knn_ppm'] = assign_ppm(df_train, grid_ppm, sector_ppm, global_ppm)
df_val['knn_ppm'] = assign_ppm(df_val, grid_ppm, sector_ppm, global_ppm)

model_features = FEATURES + ['knn_ppm']

y_train = np.log1p(df_train['price'].values)
y_val = df_val['price'].values

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[model_features].to_dict(orient='records'))
X_val = dv.transform(df_val[model_features].to_dict(orient='records'))

(X_train.shape, X_val.shape)

((2692, 55), (659, 55))

### The candidates

From dumbest to fanciest, so every better model has to justify itself against the one below it:

- **median EUR/m2 × area** - the dumbest thing that could possibly work. One price per square meter for the whole city. This is the floor everyone must beat.
- **linear regression / ridge** - cheap, linear, interpretable. How far does a straight line get us?
- **random forest, gradient boosting** (sklearn) - the tree ensembles.
- **xgboost** - the model `train.py` actually uses, and the one on trial here.

In [3]:
def score(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MAPE (%)": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
        "R2": r2_score(y_true, y_pred),
    }

predictions = {}

In [4]:
ppm_median = (df_train.price / df_train.area).median()
predictions['median EUR/m2 * area'] = ppm_median * df_val['area'].values
print(f'the dumbest of the dumbest charges {ppm_median:.2f} EUR/m2 for everything')

the dumbest of the dumbest charges 11.70 EUR/m2 for everything


In [5]:
xgb_params = dict(n_estimators=500, max_depth=5, learning_rate=0.1,
                  subsample=0.8, colsample_bytree=0.9, min_child_weight=8,
                  random_state=42) # picked some random values, we'll see best ones soon

models = {
    "linear regression": LinearRegression(),
    "ridge": Ridge(alpha=1.0),
    "random forest": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=1, max_features=0.5, random_state=42, n_jobs=-1),
    "gradient boosting": GradientBoostingRegressor(random_state=42),
    "xgboost": XGBRegressor(**xgb_params),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = np.expm1(model.predict(X_val))

results = pd.DataFrame({name: score(y_val, p) for name, p in predictions.items()}).T
results.sort_values("RMSE").round(2)

,RMSE,MAE,MAPE (%),R2
xgboost,191.14,136.50,16.37,0.77
random forest,193.68,138.25,16.54,0.77
gradient boosting,212.97,153.64,18.18,0.72
linear regression,253.98,176.45,20.88,0.60
ridge,254.89,177.16,20.96,0.60
median EUR/m2 * area,287.54,212.96,25.80,0.49


A few things stand out here. Every real model crushes the dumb baseline (290 -> ~190 RMSE for the trees), and even plain linear regression explains 60% of the variance - so the features are doing real work. The trees pull clearly ahead of the linear models.

At the top XGBoost edges random forest by ~3 EUR on this split. But this is a **single** 80/20 split, so that gap could easily be noise - I wouldn't read anything into the ordering yet. Whether either really leads is exactly what cross-validation is for. On to that next, then tuning.

In [6]:
def cross_valid_rmse(make_model, n_splits=5):
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    rmses = []

    for train_idx, val_idx in kfold.split(df):
        d_tr, d_va = df.iloc[train_idx].copy(), df.iloc[val_idx].copy()

        # fit the trim + imputation on this fold's train rows only, then apply
        # to the held-out rows - no validation row leaks into the preprocessing
        d_tr, bounds = trim_ppm_outliers(d_tr)
        d_va, _ = trim_ppm_outliers(d_va, bounds)
        d_tr, medians = impute_medians(d_tr)
        d_va, _ = impute_medians(d_va, medians)

        grid_ppm, sector_ppm, global_ppm = build_ppm_grid(d_tr)
        d_tr['knn_ppm'] = assign_ppm(d_tr, grid_ppm, sector_ppm, global_ppm)
        d_va['knn_ppm'] = assign_ppm(d_va, grid_ppm, sector_ppm, global_ppm)

        # the market-rate trim, exactly like train.py: fixed 0.5x-2x band,
        # then the neighbor rate gets recomputed from the trimmed train rows
        d_tr = trim_knn_outliers(d_tr).copy()
        d_va = trim_knn_outliers(d_va).copy()
        grid_ppm, sector_ppm, global_ppm = build_ppm_grid(d_tr)
        d_tr['knn_ppm'] = assign_ppm(d_tr, grid_ppm, sector_ppm, global_ppm)
        d_va['knn_ppm'] = assign_ppm(d_va, grid_ppm, sector_ppm, global_ppm)
        
        dv_ = DictVectorizer(sparse=False)
        X_tr = dv_.fit_transform(d_tr[model_features].to_dict(orient='records'))
        X_va = dv_.transform(d_va[model_features].to_dict(orient='records'))
        m = make_model()
        
        m.fit(X_tr, np.log1p(d_tr["price"].values))
        cv_preds = np.expm1(m.predict(X_va))
        rmses.append(np.sqrt(mean_squared_error(d_va["price"].values, cv_preds)))
        
    return np.mean(rmses), np.std(rmses)

factories = {
    "linear regression": lambda: LinearRegression(),
    "random forest": lambda: RandomForestRegressor(
        n_estimators=300, min_samples_leaf=1, max_features=0.5, random_state=42, n_jobs=-1),
    "gradient boosting": lambda: GradientBoostingRegressor(random_state=42),
    "xgboost": lambda: XGBRegressor(**xgb_params),
}

for name, factory in factories.items():
    mean, std = cross_valid_rmse(factory)
    print(f"{name}  RMSE {mean:4.0f} +- {std:.0f} EUR")

linear regression  RMSE  249 +- 14 EUR
random forest  RMSE  200 +- 5 EUR
gradient boosting  RMSE  214 +- 4 EUR
xgboost  RMSE  198 +- 6 EUR


Averaged over 5 folds the two stay locked together: XGBoost at **198 +- 6 EUR**, random forest just behind at 200 +- 5 - the same order as the single split, but with the +- bands overlapping completely. Which is exactly why I cross-validate: on a gap this small, one split alone will happily exaggerate who's winning.

And remember these XGBoost numbers come from parameters I picked at random. If it's already matching a decently-configured random forest while untuned, the search should tell us who really wins. Let's find out.

In [7]:
rng = np.random.default_rng(42)

def random_xgb_config():
    return {
        "n_estimators": int(rng.choice([200, 300, 400, 600, 800])),
        "max_depth": int(rng.integers(3, 9)),
        "learning_rate": float(rng.choice([0.02, 0.03, 0.05, 0.08, 0.12])),
        "subsample": round(float(rng.uniform(0.7, 1.0)), 2),
        "colsample_bytree": round(float(rng.uniform(0.7, 1.0)), 2),
        "min_child_weight": int(rng.integers(1, 9)),
    }

xgb_results = []
for i in range(10):
    cfg = random_xgb_config()
    mean, std = cross_valid_rmse(lambda: XGBRegressor(random_state=42, **cfg))
    xgb_results.append({**cfg, "cv_rmse": round(mean, 1), "std": round(std, 1)})
    print(f"{i + 1:2d}/10  RMSE {mean:5.1f} +- {std:4.1f}  {cfg}")

pd.DataFrame(xgb_results).sort_values("cv_rmse").head(5)

 1/10  RMSE 193.9 +-  7.0  {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.08, 'subsample': 0.96, 'colsample_bytree': 0.91, 'min_child_weight': 4}
 2/10  RMSE 207.6 +-  3.7  {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.05, 'subsample': 0.93, 'colsample_bytree': 0.94, 'min_child_weight': 8}
 3/10  RMSE 197.2 +-  6.0  {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.12, 'subsample': 0.81, 'colsample_bytree': 0.98, 'min_child_weight': 4}
 4/10  RMSE 192.5 +-  6.5  {'n_estimators': 600, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.83, 'colsample_bytree': 0.77, 'min_child_weight': 7}
 5/10  RMSE 196.6 +-  5.6  {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.12, 'subsample': 0.95, 'colsample_bytree': 0.89, 'min_child_weight': 1}
 6/10  RMSE 194.3 +-  5.0  {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.08, 'subsample': 0.99, 'colsample_bytree': 0.97, 'min_child_weight': 3}
 7/10  RMSE 192.4 +-  6.2  {'n_estimators': 600, 'max_depth': 7,

,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,cv_rmse,std
6,600,7,0.08,0.84,0.71,2,192.4,6.2
3,600,6,0.05,0.83,0.77,7,192.5,6.5
0,200,7,0.08,0.96,0.91,4,193.9,7.0
5,200,7,0.08,0.99,0.97,3,194.3,5.0
8,400,4,0.12,0.84,0.76,3,195.5,4.7


### Tuning random forest too, to keep it fair

It'd be cheap to tune only XGBoost and declare victory. Since random forest was the one breathing down its neck, it gets a proper search as well - otherwise the comparison isn't honest.

In [8]:
rf_grid = [
    {"n_estimators": n, "min_samples_leaf": leaf, "max_features": mf}
    for n in [300] for leaf in [1, 3, 5] for mf in [0.5, 1.0]
]

rf_results = []
for cfg in rf_grid:
    mean, std = cross_valid_rmse(lambda: RandomForestRegressor(random_state=42, n_jobs=-1, **cfg))
    rf_results.append({**cfg, "cv_rmse": round(mean, 1), "std": round(std, 1)})
    print(f"RMSE {mean:5.1f} +- {std:4.1f}  {cfg}")

pd.DataFrame(rf_results).sort_values("cv_rmse")

RMSE 199.8 +-  5.0  {'n_estimators': 300, 'min_samples_leaf': 1, 'max_features': 0.5}
RMSE 201.2 +-  4.2  {'n_estimators': 300, 'min_samples_leaf': 1, 'max_features': 1.0}
RMSE 203.4 +-  4.6  {'n_estimators': 300, 'min_samples_leaf': 3, 'max_features': 0.5}
RMSE 203.9 +-  4.5  {'n_estimators': 300, 'min_samples_leaf': 3, 'max_features': 1.0}
RMSE 207.2 +-  4.7  {'n_estimators': 300, 'min_samples_leaf': 5, 'max_features': 0.5}
RMSE 206.9 +-  5.2  {'n_estimators': 300, 'min_samples_leaf': 5, 'max_features': 1.0}


,n_estimators,min_samples_leaf,max_features,cv_rmse,std
0,300,1,0.5,199.8,5.0
1,300,1,1.0,201.2,4.2
2,300,3,0.5,203.4,4.6
3,300,3,1.0,203.9,4.5
5,300,5,1.0,206.9,5.2
4,300,5,0.5,207.2,4.7


### Conclusion! XGBoost wins, and tuning widens the gap

The two searches settle the near-tie from cross-validation. Tuning helped XGBoost and did almost nothing for random forest:

| model | untuned CV RMSE | best tuned CV RMSE |
|---|---|---|
| xgboost | 198 +- 6 | 192.4 +- 6.2 |
| random forest | 200 +- 5 | 199.8 +- 5.0 |

- XGBoost dropped from 198 to 192.4 EUR. The best draw was `n_estimators=600, max_depth=7, learning_rate=0.08, subsample=0.84, colsample_bytree=0.71, min_child_weight=2`, but the config `train.py` actually ships (`600, max_depth=6, lr=0.05, subsample=0.83, colsample_bytree=0.77, min_child_weight=7`) lands at 192.5, within 0.1 EUR of it - a dead heat, so there's no reason to switch. Mild regularization keeps paying off on this ~3.4k-row dataset.
- Random forest basically didn't move: its default-ish config (`min_samples_leaf=1, max_features=0.5`) was already the best of the grid at 199.8, and every other setting was worse. Little headroom left to tune.

So XGBoost leads both untuned and tuned, and the search only widens the gap to about 7 EUR. The winning region of the search is essentially the config `train.py` already ships, and nothing here argues for switching.

the random search was only 10 draws, so 192.4 is a floor-ish estimate, not the true optimum - a wider search would likely shave off a bit more. Next step is to lock in these params and retrain the final model on the full data.